# SupeRR Selector Season 2 — Consolidated Handoff (Phases 0–3)

This notebook preserves the validated phase boundary: Phase 3 starts from the Phase 2 master. It does not reload the six raw CSVs or repeat Phase 1/2 cleaning. The former score/target/required-rate Phase 3 is superseded because exact legal-ball and score reconstruction is not supported by the available evidence.

## Phase 0 and Phase 1 decisions

- Pressure is pre-delivery and identical for batter and bowler on the same record.
- IPL and T20I retain separate baselines.
- `player_id` is the identity key.
- The approved completed population contains 813 matches: 362 IPL and 451 T20I.
- Unsupported cricket states are not imputed.

In [ ]:
from pathlib import Path
import pandas as pd

from phase3_structural_pressure import run_pipeline

ROOT = Path.cwd()
PHASE2_MASTER = ROOT / 'phase2_outputs' / 'phase2_integrated_cleaned_deliveries.csv'
PHASE3_OUTPUT_DIR = ROOT / 'phase3_outputs'
PHASE3_REPORT = ROOT / 'Phase 3 Structural Pressure Report.md'

phase2 = pd.read_csv(PHASE2_MASTER, low_memory=False)
phase2_summary = pd.DataFrame([{
    'regulation_delivery_records': len(phase2),
    'eligible_matches': phase2['match_id'].nunique(),
    'regulation_innings': phase2[['match_id', 'innings']].drop_duplicates().shape[0],
    'columns': len(phase2.columns),
    'duplicate_delivery_keys': phase2.duplicated(['match_id', 'innings', 'innings_ball_num']).sum(),
}])
phase2_summary

## Phase 2 limitation carried forward

Phase 2 contains 7,976 records with recorded wide/overstep evidence, 184,002 records with no recorded wide/overstep, and 257 unresolved records. It also records 147 extended overs without a recorded wide/overstep. These facts prohibit an exact inferred legal-ball counter. Runs/extras semantics are also insufficient for a claimed complete team-score reconstruction.

In [ ]:
legal_evidence_summary = phase2['legal_ball_evidence_status'].value_counts(dropna=False).rename_axis('status').reset_index(name='records')
legal_evidence_summary

## Corrected Phase 3

The structural-pressure model excludes whole revised/shortened matches and uses only the supplied source over, wickets already lost before the current record, prior-record wicket shock, and leave-one-match-out expectations. It does not use scores, targets, required rates, inferred legal balls, balls remaining, or the current-delivery outcome.

In [ ]:
results = run_pipeline(PHASE2_MASTER, PHASE3_OUTPUT_DIR, PHASE3_REPORT)
pd.DataFrame([{
    'primary_delivery_records': len(results['deliveries']),
    'primary_matches': results['deliveries']['match_id'].nunique(),
    'excluded_revised_matches': len(results['excluded_matches']),
    'minimum_benchmark_support': results['benchmarks']['benchmark_support'].min(),
    'maximum_benchmark_support': results['benchmarks']['benchmark_support'].max(),
}])

## Acceptance gates, robustness, and manifest

In [ ]:
results['validation']

In [ ]:
results['sensitivity']

In [ ]:
results['manifest']

## Phase 4 handoff

Use only `phase3_outputs/phase3_structural_pressure_labelled.csv`. Retain the continuous index for primary estimation, preserve the same pre-delivery pressure state for batter and bowler, keep revised matches out of the primary analysis, and separately feasibility-audit every outcome metric that depends on runs or extras.